# Urejanje besedilnih podatkov in regularni izrazi

Spletna enciklopedija [Wikipedia](https://www.wikipedia.org/) je bogat vir podatkov, na primer [tabel s podatki o bruto domačem proizvodu držav](https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(nominal)).

Spoznali bomo uvoz podatkov s spletnih strani in urejanje uvoženega besedila. Pri obdelavi in čiščenju nizov znakov si bomo pomagali z **regularnimi izrazi**.

## Uvoz spletnih strani: knjižnici `requests` in `bs4`

Spletno stran prenesemo s funkcijo [`requests.get`](https://requests.readthedocs.io/en/latest/user/quickstart/#make-a-request). Njen prvi argument je [URL](https://en.wikipedia.org/wiki/URL), torej spletni naslov. Preberimo [članek o Univerzi v Ljubljani](https://sl.wikipedia.org/wiki/Univerza_v_Ljubljani). Z argumentom `timeout` omejimo čakanje na odziv strežnika, v `headers` pa podamo glavo `User-Agent`, saj nekateri strežniki zavračajo zahteve brez ustrezne oznake odjemalca:

In [1]:
import requests

headers = {"User-Agent": "Mozilla/5.0"}
html = requests.get(
    "https://sl.wikipedia.org/wiki/Univerza_v_Ljubljani",
    headers=headers,
    timeout=30
)
print(html)

<Response [200]>


Rezultat je objekt razreda [`Response`](https://requests.readthedocs.io/en/latest/api/#requests.Response). Oglejmo si atributa `status_code` in `content`:

In [2]:
print(html.status_code)
print(html.content)
html.raise_for_status()

200
b'<!DOCTYPE html>\n<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientpref-day vector-sticky-header-enabled vector-toc-available skin-thumbsize-clientpref-standard skin-underline-clientpref-browser" lang="sl" dir="ltr">\n<head>\n<meta charset="UTF-8">\n<title>Univerza v Ljubljani - Wikipedija, prosta enciklopedija</title>\n<script>(function(){var className="client-js vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feat

Atribut `status_code` vsebuje statusno kodo HTTP. Vrednost `200` pomeni uspešen odziv. Kode od `400` do `499` označujejo napake odjemalca: `404` na primer pomeni, da strežnik ni našel zahtevane strani, `403` pa, da je dostop zavrnjen. Seznam kod in pomenov je v članku [List of HTTP status codes](https://en.wikipedia.org/wiki/List_of_HTTP_status_codes).

Metoda `raise_for_status()` ob neuspešnem odzivu s kodo od `400` do `599` sproži izjemo. Tako preprečimo, da bi stran z obvestilom o napaki obdelovali kot podatke.

Atribut `content` vsebuje **bajte** izvorne kode strani v označevalnem jeziku [HTML](https://en.wikipedia.org/wiki/HTML). Atribut `text` vsebuje isto vsebino, dekodirano v niz znakov.

Iz članka želimo pridobiti imena članic Univerze v Ljubljani in jih shraniti v zaporedje knjižnice `pandas`. Za izbiro elementov HTML potrebujemo razčlenjevalnik (angl. _parser_). Ustvarimo ga s konstruktorjem [`BeautifulSoup`](https://www.crummy.com/software/BeautifulSoup/bs4/doc/#making-the-soup) iz knjižnice `bs4`:

In [3]:
from bs4 import BeautifulSoup

juha = BeautifulSoup(html.content, "html.parser")
print(type(juha))
print(juha)

<class 'bs4.BeautifulSoup'>
<!DOCTYPE html>

<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientpref-day vector-sticky-header-enabled vector-toc-available skin-thumbsize-clientpref-standard skin-underline-clientpref-browser" dir="ltr" lang="sl">
<head>
<meta charset="utf-8"/>
<title>Univerza v Ljubljani - Wikipedija, prosta enciklopedija</title>
<script>(function(){var className="client-js vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-d

Objekt razreda `BeautifulSoup` omogoča dostop do posameznih elementov strani. Metoda [`select`](https://www.crummy.com/software/BeautifulSoup/bs4/doc/#css-selectors) jih izbere z izbirnikom CSS (angl. _CSS selector_).

Izbrati želimo seznam v razdelku _Trenutne članice_. Izvorno kodo tega dela strani si v brskalniku ogledamo z ukazom **Preglej** oziroma **Inspect** v meniju, ki ga odpremo z desnim klikom na izbrani element. Ustrezni del kode ima naslednjo obliko:

```html
<div class="references-small" style="column-count:3;">
    <ul>
        <li><a href="/wiki/Akademija_za_glasbo_v_Ljubljani">Akademija za glasbo</a></li>
        ...
    </ul>
</div>
```

Seznam je v elementu `div` z razredom CSS `references-small`. Izbirnik `div.references-small` zato izbere vse take elemente. Zgradba spletnih strani se lahko spremeni; če rezultat ni pričakovan, ponovno preverimo izvorno kodo in izbirnik:

In [4]:
nastevni_seznam_html = juha.select("div.references-small")
print(nastevni_seznam_html)

[<div about="#mwt67" class="references-small" data-mw='{"parts":[{"template":{"target":{"wt":"stolpci","href":"./Predloga:Stolpci"},"params":{"1":{"wt":"3"},"2":{"wt":"\n* [[Akademija za glasbo v Ljubljani|Akademija za glasbo]]\n* [[Akademija za gledališče, radio, film in televizijo v Ljubljani|Akademija za gledališče, radio, film in televizijo]]\n* [[Akademija za likovno umetnost in oblikovanje v Ljubljani|Akademija za likovno umetnost in oblikovanje]]\n* [[Biotehniška fakulteta v Ljubljani|Biotehniška fakulteta]]\n* [[Ekonomska fakulteta v Ljubljani|Ekonomska fakulteta]]\n* [[Fakulteta za arhitekturo v Ljubljani|Fakulteta za arhitekturo]]\n* [[Fakulteta za družbene vede v Ljubljani|Fakulteta za družbene vede]]\n* [[Fakulteta za elektrotehniko v Ljubljani|Fakulteta za elektrotehniko]]\n* [[Fakulteta za farmacijo v Ljubljani|Fakulteta za farmacijo]]\n* [[Fakulteta za gradbeništvo in geodezijo v Ljubljani|Fakulteta za gradbeništvo in geodezijo]]\n* [[Fakulteta za kemijo in kemijsko tehn

Rezultat metode `select` je seznam elementov HTML. Med zadetki je tudi seznam nekdanjih članic, zato preverimo vsebino. V uporabljeni različici strani prvi zadetek vsebuje trenutne članice:

In [5]:
nastevni_seznam_fakultet = nastevni_seznam_html[0]
print(nastevni_seznam_fakultet)

<div about="#mwt67" class="references-small" data-mw='{"parts":[{"template":{"target":{"wt":"stolpci","href":"./Predloga:Stolpci"},"params":{"1":{"wt":"3"},"2":{"wt":"\n* [[Akademija za glasbo v Ljubljani|Akademija za glasbo]]\n* [[Akademija za gledališče, radio, film in televizijo v Ljubljani|Akademija za gledališče, radio, film in televizijo]]\n* [[Akademija za likovno umetnost in oblikovanje v Ljubljani|Akademija za likovno umetnost in oblikovanje]]\n* [[Biotehniška fakulteta v Ljubljani|Biotehniška fakulteta]]\n* [[Ekonomska fakulteta v Ljubljani|Ekonomska fakulteta]]\n* [[Fakulteta za arhitekturo v Ljubljani|Fakulteta za arhitekturo]]\n* [[Fakulteta za družbene vede v Ljubljani|Fakulteta za družbene vede]]\n* [[Fakulteta za elektrotehniko v Ljubljani|Fakulteta za elektrotehniko]]\n* [[Fakulteta za farmacijo v Ljubljani|Fakulteta za farmacijo]]\n* [[Fakulteta za gradbeništvo in geodezijo v Ljubljani|Fakulteta za gradbeništvo in geodezijo]]\n* [[Fakulteta za kemijo in kemijsko tehno

Izbrani element je objekt razreda `bs4.element.Tag`. Postavke neurejenega seznama `ul` so označene z `li`. Z metodo [`find_all`](https://www.crummy.com/software/BeautifulSoup/bs4/doc/#find-all) poiščemo vse take postavke:

In [6]:
fakultete_html = nastevni_seznam_fakultet.find_all("li")
print(fakultete_html)

[<li><a class="mw-redirect" href="https://sl.wikipedia.org/wiki/Akademija_za_glasbo_v_Ljubljani" rel="mw:WikiLink" title="Akademija za glasbo v Ljubljani">Akademija za glasbo</a></li>, <li><a href="https://sl.wikipedia.org/wiki/Akademija_za_gledališče,_radio,_film_in_televizijo_v_Ljubljani" rel="mw:WikiLink" title="Akademija za gledališče, radio, film in televizijo v Ljubljani">Akademija za gledališče, radio, film in televizijo</a></li>, <li><a href="https://sl.wikipedia.org/wiki/Akademija_za_likovno_umetnost_in_oblikovanje_v_Ljubljani" rel="mw:WikiLink" title="Akademija za likovno umetnost in oblikovanje v Ljubljani">Akademija za likovno umetnost in oblikovanje</a></li>, <li><a href="https://sl.wikipedia.org/wiki/Biotehniška_fakulteta_v_Ljubljani" rel="mw:WikiLink" title="Biotehniška fakulteta v Ljubljani">Biotehniška fakulteta</a></li>, <li><a href="https://sl.wikipedia.org/wiki/Ekonomska_fakulteta_v_Ljubljani" rel="mw:WikiLink" title="Ekonomska fakulteta v Ljubljani">Ekonomska fakul

Vsak element dobljenega seznama predstavlja eno članico univerze. Z atributom `text` ohranimo le besedilo brez oznak HTML:

In [7]:
fakultete = [f.text for f in fakultete_html]
print(fakultete)

['Akademija za glasbo', 'Akademija za gledališče, radio, film in televizijo', 'Akademija za likovno umetnost in oblikovanje', 'Biotehniška fakulteta', 'Ekonomska fakulteta', 'Fakulteta za arhitekturo', 'Fakulteta za družbene vede', 'Fakulteta za elektrotehniko', 'Fakulteta za farmacijo', 'Fakulteta za gradbeništvo in geodezijo', 'Fakulteta za kemijo in kemijsko tehnologijo', 'Fakulteta za matematiko in fiziko', 'Fakulteta za pomorstvo in promet', 'Fakulteta za računalništvo in informatiko', 'Fakulteta za socialno delo', 'Fakulteta za strojništvo', 'Fakulteta za šport', 'Fakulteta za upravo', 'Filozofska fakulteta', 'Medicinska fakulteta', 'Naravoslovno-tehniška fakulteta', 'Pedagoška fakulteta', 'Pravna fakulteta', 'Teološka fakulteta', 'Veterinarska fakulteta', 'Zdravstvena fakulteta']


Seznam imen članic s konstruktorjem [`Series`](https://pandas.pydata.org/docs/reference/api/pandas.Series.html) pretvorimo v zaporedje:

In [8]:
import pandas as pd

fakultete = pd.Series(fakultete)
print(fakultete)

0                                   Akademija za glasbo
1     Akademija za gledališče, radio, film in televi...
2          Akademija za likovno umetnost in oblikovanje
3                                 Biotehniška fakulteta
4                                   Ekonomska fakulteta
5                              Fakulteta za arhitekturo
6                            Fakulteta za družbene vede
7                           Fakulteta za elektrotehniko
8                                Fakulteta za farmacijo
9                Fakulteta za gradbeništvo in geodezijo
10          Fakulteta za kemijo in kemijsko tehnologijo
11                    Fakulteta za matematiko in fiziko
12                     Fakulteta za pomorstvo in promet
13            Fakulteta za računalništvo in informatiko
14                           Fakulteta za socialno delo
15                             Fakulteta za strojništvo
16                                   Fakulteta za šport
17                                  Fakulteta za

Pozneje bomo iz imen članic sestavili kratice, na primer FMF za Fakulteto za matematiko in fiziko. Pred tem spoznajmo še uvoz spletnih tabel.

## Uvoz tabelaričnih podatkov s spleta

Funkcija [`read_html`](https://pandas.pydata.org/docs/reference/api/pandas.read_html.html) prebere tabele iz vsebine HTML in vrne seznam podatkovnih tabel. Prenesimo članek o nominalnem bruto domačem proizvodu držav. Besedilo odziva ovijemo v `StringIO`, da ga podamo kot datoteki podoben objekt:

In [9]:
from io import StringIO

bdp_html = requests.get(
    "https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(nominal)",
    headers=headers,
    timeout=30
)
bdp_html.raise_for_status()
tabele = pd.read_html(StringIO(bdp_html.text))
print(len(tabele))

8


Vrstni red tabel na strani se lahko spremeni, zato tabele z BDP ne izberemo po fiksnem položaju. Z argumentom `match` izberemo tabele z besedilom `Country/Territory`, ki se pojavi v glavi želene tabele:

In [10]:
tabele_bdp = pd.read_html(StringIO(bdp_html.text), match="Country/Territory")
tabela = tabele_bdp[0]
print(tabela)

    Country/Territory IMF (2026)[1] World Bank (2025)[6]  \
0               World     128003776            118350166   
1       United States      32642373             30769700   
2          China[n 1]      21546403             19498039   
3             Germany       5471299              5050923   
4               Japan       4341986              4435163   
..                ...           ...                  ...   
217             Palau           377                  345   
218  Marshall Islands           342                  308   
219             Nauru           196                  176   
220        Montserrat          —N/a                 —N/a   
221            Tuvalu            65                   57   

    United Nations (2024)[7]  
0                  111565144  
1                   29298000  
2                   18743802  
3                    4659929  
4                    4026211  
..                       ...  
217                      310  
218                      281  


Uvožena tabela lahko poleg podatkov vsebuje oznake opomb in manjkajoče vrednosti, zapisane z znakom `—`. V starejših različicah strani so bile v stolpcih z letnicami tudi vrednosti, kot je `[n 3]2022`; v novejših različicah so letnice lahko del glave tabele. Pred analizo zato pregledamo stolpce in podatke **prečistimo**. Pri tem nam pomagajo regularni izrazi.

## Regularni izrazi

Regularni izraz (angl. _regular expression_) je poseben niz znakov, ki ga uporabljamo za kratek zapis cele množice običajnih nizov znakov. Za podan regularni izraz $r$ z $L(r)$ označimo množico nizov znakov, ki jih regularni izraz zapiše. Množico $L(r)$ imenujemo _jezik regularnega izraza_. Rečemo tudi, da regularni izraz $r$ _tvori_ nize znakov iz jezika $L(r)$. Za vsak niz znakov iz $L(r)$ rečemo, da se _ujema z regularnim izrazom_ ali pa _ustreza regularnemu izrazu_ $r$.

Regularni izraz lahko tvorimo s pomočjo treh operatorjev.

1. Operator **stika** $\cdot$. Za množico nizov znakov $L(r_1 \cdot r_2)$, ki jih tvori stik dveh regularnih izrazov $r_1$ in $r_2$, velja

    $$ L(r_1 \cdot r_2) = \{ s_1 \cdot s_2 : s_1 \in L(r_1) \land s_2 \in L(r_2) \}, $$

    kjer je $s_1 \cdot s_2$ niz znakov, ki ga dobimo tako, da zapišemo oba niza znakov $s_1$ in $s_2$ en za drugim. Jezik, ki ga definira stik regularnih izrazov $\text{abc} \cdot \text{def}$, vsebuje zgolj en niz znakov $\text{abcdef}$.
    V regularnih izrazih pogosto operator stika ne pišemo, podobno kot v zapisu matematičnih izrazov ne pišemo operatorja množenja.

1. Operator **unije** (izbire) $|$. Za množico nizov znakov $L(r_1 | r_2)$, ki jih tvori unija dveh regularnih izrazov $r_1$ in $r_2$, velja

    $$ L(r_1 | r_2) = \{ s : s \in L(r_1) \lor s \in L(r_2) \} = L(r_1) \cup L(r_2). $$

    Z drugimi besedami, v jeziku unije dveh regularnih izrazov so nizi znakov, ki pripadajo jeziku vsaj enega izmed njih. Jezik regularnega izraza $\text{abc}|\text{def}$ je tako množica nizov znakov $\{ \text{abc}, \text{def} \}$.

1. Operator **iteracije** (ponavljanja) $^*$. Za množico nizov znakov $L(r^*)$, ki jih tvori iteracija regularnega izraza $r$, velja

    $$ L(r^*) = \{ s_1 \cdot s_2 \cdot \ldots \cdot s_n : n \geq 0 \land s_i \in L(r) \}. $$
    
    V jeziku iteracije regularnega izraza so torej nizi znakov, ki jih lahko zapišemo kot stik poljubnega števila nizov znakov iz jezika $L(r)$. Poljubno število je tudi 0, kar pomeni, da je v jeziku $L(r^*)$ vedno prazen niz znakov, ki ga označimo z $\epsilon$. Tako je, na primer, jezik regularnega izraza $(\text{abc})^*$ neskončna množica nizov znakov $\{ \epsilon, \text{abc}, \text{abcabc}, \text{abcabcabc}, \ldots \}$.

Najvišjo prednost ima operator iteracije, temu sledi stik in na koncu operator unije. Na primer, $L(\text{ab}|\text{c}^*) = \{ ab, \epsilon, c, cc, ccc, \ldots \}$, po drugi strani pa $L(\text{a}(\text{b}|\text{c})^*) = \{ a, ab, ac, abb, abc, acb, acc, \ldots \} $.

Zanimive množice nizov znakov dobimo takrat, ko začnemo kombinirati te tri operatorje. Oglejmo si nekaj primerov regularnih izrazov za jezike nizov znakov, sestavljenih iz znakov 0 in 1.

* $(0|1)^*$: poljuben niz znakov 0 in 1;
* $101(0|1)^*$: poljuben niz znakov 0 in 1 s predpono 101;
* $(0|1)^*(010|101)(0|1)^*$: poljuben niz znakov 0 in 1, ki vsebuje enega od nizov 010 in 101;
* $(00)^*$: poljuben niz sodega števila znakov 0;
* $(00|01|10|11)^*$: poljuben niz sodega števila znakov 0 ali 1;
* $(0|1)(00|01|10|11)^*$: poljuben niz lihega števila znakov 0 ali 1.

### Regularni izrazi v Pythonu

Regularne izraze v Pythonu pišemo med enojnimi ali dvojnimi narekovaji, tako kot običajne nize znakov, a z dodatno predpono `r` (pomen slednje bomo pojasnili pozneje). Operatorja _stika_ ne pišemo (zaporedno zapisana dela izraza sta v stiku), za operatorja _unije_ in _iteracije_ pa uporabljamo znaka `|` in `*`. Python poleg omenjenih treh operatorjev ponuja še vrsto drugih simbolov, ki nam olajšajo sestavljanje regularnih izrazov. Razdelimo jih lahko v tri kategorije, opisane v nadaljevanju.

Prva kategorija so posebni regularni izrazi, ki nam omogočajo kratek zapis **množice znakov**.

* `.` je regularni izraz, ki se ujema s poljubnim znakom, razen z znakom za novo vrstico `\n`. Če hočemo zapisati regularni izraz, ki se ujema zgolj z znakom za piko `.`, moramo znaku `.` dodati predpono `\`: zapišemo ga torej `\.`.

* `[...]` se ujema s poljubnim znakom iz `...`. Posebna varianta tega regularnega izraza je `[^...]`, ki se ujema s poljubnim znakom, ki ga ni v `...`. Seznam lahko krajšamo z uporabo znaka `-`, ki določa razpon znakov. Nekaj primerov:

  * `[0123456789]` in `[0-9]` se ujemata s poljubno števko ASCII;
  * `[a-z]` in `[A-Z]` se ujemata z malo oziroma veliko črko angleške abecede;
  * `[0-46-9]` se ujema s poljubno števko, razen s števko 5;
  * `[^0-9]` se ujema z znakom, ki ni števka ASCII od 0 do 9;
  * `[aeiouAEIOU]` se ujema s poljubnim samoglasnikom;
  * `[^aeiouAEIOU]` se ujema s poljubnim znakom, ki ni eden od navedenih samoglasnikov. To vključuje tudi števke in ločila, zato izraz sam po sebi ne prepoznava zgolj soglasnikov.

* `\d` se ujema z desetiško števko Unicode. Izraz `[0-9]` se omeji na števke ASCII od 0 do 9.

* `\D` se ujema z znakom, ki ni desetiška števka Unicode.

* `\s` se ujema z belim znakom, na primer s presledkom, tabulatorjem ali znakom za novo vrstico. Pri nizih Unicode vključuje tudi druge bele znake.

* `\S` se ujema z znakom, ki ni bel znak.

* `\w` se ujema s črko ali števko Unicode oziroma s podčrtajem `_`. Vključuje tudi črke č, š in ž; `[a-zA-Z0-9_]` se omeji na navedene znake ASCII.

* `\W` se ujema z znakom, ki ne ustreza `\w`.

Druga kategorija so simboli za posebne vrste iteracij, ki jih imenujemo **kvantifikatorji** in omogočajo **omejevanje števila ponovitev v iteraciji**.

| Kvantifikator | Število ponovitev $n$ |
|:---|:---|
| `*` | $n \geq 0$ |
| `+` | $n \geq 1$ |
| `?` | $n \in \{0, 1\}$ |
| `{x}` | $n = x$ |
| `{x,}` | $n \geq x$ |
| `{,y}` | $0 \leq n \leq y$ |
| `{x,y}` | $x \leq n \leq y$ |

Tretja kategorija so **različni uporabni simboli**.

* `^` se ujema z začetkom niza znakov.

* `$` se ujema s koncem niza ali s položajem tik pred zaključno novo vrstico. Za preverjanje celotnega niza je priročna metoda [`fullmatch`](https://docs.python.org/3/library/re.html#re.Pattern.fullmatch).

### Zajem podnizov

Z regularnimi izrazi lahko tudi _zajamemo_ podniz znakov, ki se ujema z delom regularnega izraza. Del regularnega izraza za zajem zamejimo z navadnimi oklepaji `(` in `)`. Zajemne skupine številčimo po vrstnem redu začetnih oklepajev od 1 dalje. V vzorcu in nadomestnem nizu se nanje sklicujemo z `\1`, `\2` itd.; iz objekta ujemanja jih pridobimo z metodo `group`.

Pythonov modul `re` ne podpira razredov POSIX, kot je `[:alpha:]`. V spodnjih primerih zato črke navedemo izrecno; dodamo tudi slovenske šumnike.

Zajeto skupino znakov lahko uporabimo pri definiciji regularnega izraza zato, da se sklicujemo na del niza znakov, ki se ujema z delom za zajem. V nadaljevanju podajamo nekaj primerov. Pravo moč zajetih nizov znakov bomo videli potem, ko bomo spoznali funkcijo [`re.sub`](https://docs.python.org/3/library/re.html#re.sub) v naslednjem razdelku.

* `([a-zA-ZčšžČŠŽ])[a-zA-ZčšžČŠŽ]*\1` se ujema s katerimkoli nizom vsaj dveh navedenih črk, kjer je prva črka enaka zadnji. Zajeti niz znakov se sestoji iz prve črke, ki je zamejena z oklepaji, nato sledi poljubno število črk in na koncu se mora pojaviti zajeti niz znakov. Ta je enak prvi črki, torej zadnja črka mora biti enaka prvi.

* `(([a-zA-ZčšžČŠŽ]+) (\d{4}))` v nizu znakov `maja 2014` zajame tri skupine `\1 = maja 2014`, `\2 = maja` in `\3 = 2014`. Zajame torej celoten zapis meseca in leta, ime meseca ter letnico.

* `(\d+)x(\d+)` se ujema z nizi znakov, ki zapišejo dimenzije matrike, na primer, `2x3` ali `302x11`. Pri tem je prva zajeta skupina `\1` enaka prvi komponenti dimenzije, torej `2` oziroma `302` v omenjenih primerih, drugo zajetje `\2` pa drugi komponenti, `3` oziroma `11`.

### Funkcije in metode modula `re`

Delo z regularnimi izrazi v Pythonu nam omogoča standardni modul [`re`](https://docs.python.org/3/howto/regex.html). V nadaljevanju bomo spoznali nekaj njegovih funkcij in metod.

S funkcijo [`compile`](https://docs.python.org/3/library/re.html#re.compile) vzorec prevedemo v objekt razreda `Pattern`:

In [11]:
import re

beseda = "beseda"
beseda_s_presledki = " beseda "
besedilo = "Prvi stavek je kratek. Drugi stavek omenja tudi številke, na primer, 10 in 20."

ri1 = re.compile(r"\w+")
print(ri1)

re.compile('\\w+')


Predpona `r` določa surovi niz (angl. _raw string_). Python v njem obratnih poševnic `\` ne obravnava kot začetkov ubežnih zaporedij. Tako je `r"\n"` niz dveh znakov, obratnih obratne poševnice in črke `n`, običajni niz `"\n"` pa vsebuje znak za novo vrstico.

Pomembno je ločiti zapis niza v Pythonu od njegove interpretacije v modulu `re`: regularni izraz `r"\n"` se še vedno ujema z **znakom za novo vrstico**, ker modul `re` zaporedje `\n` razume na ta način. Za iskanje dobesedne obratnih obratne poševnice in črke `n` uporabimo `r"\\n"`.

Metoda [`match`](https://docs.python.org/3/library/re.html#re.Pattern.match) preveri ujemanje na začetku niza. Vrne objekt ujemanja ali `None`:

In [12]:
u1 = ri1.match(beseda)
print(f"Objekt ujemanja: {u1}")
print(f"span(): {u1.span()}")
print(f"group(): {u1.group()}")

u2 = ri1.match(beseda_s_presledki)
print(u2)

Objekt ujemanja: <re.Match object; span=(0, 6), match='beseda'>
span(): (0, 6)
group(): beseda
None


Ob ujemanju dobimo objekt razreda [`Match`](https://docs.python.org/3/library/re.html#match-objects). Metoda `span()` vrne začetni in končni položaj zadetka, pri čemer končni položaj ni vključen. Metoda `group()` vrne ujemajoči se podniz.

Metoda [`search`](https://docs.python.org/3/library/re.html#re.Pattern.search) poišče prvi zadetek kjerkoli v nizu:

In [13]:
u3 = ri1.search(beseda_s_presledki)
print(u3)

<re.Match object; span=(1, 7), match='beseda'>


Metoda [`findall`](https://docs.python.org/3/library/re.html#re.Pattern.findall) vrne seznam vseh neprekrivajočih se zadetkov. Pri izrazu brez zajemnih skupin so to ujemajoči se podnizi:

In [14]:
print(ri1.findall(beseda))
print(ri1.findall(beseda_s_presledki))
print(ri1.findall(besedilo))

['beseda']
['beseda']
['Prvi', 'stavek', 'je', 'kratek', 'Drugi', 'stavek', 'omenja', 'tudi', 'številke', 'na', 'primer', '10', 'in', '20']


Z regularnim izrazom `ri1` besedilo razdelimo na podnize iz črk, števk in podčrtajev. Pri čiščenju besedila moramo preveriti, ali taka opredelitev besede ustreza naši nalogi.

Funkcija [`re.sub`](https://docs.python.org/3/library/re.html#re.sub) ima argumente `pattern`, `repl` in `string`: vzorec, nadomestni niz in izvirno besedilo. Vse neprekrivajoče se zadetke zamenja z nadomestnim nizom. Argument `count` omeji število zamenjav; privzeta vrednost `0` pomeni vse zadetke.

Pri prevedenem regularnem izrazu uporabimo metodo `sub`, ki ji vzorca ne podamo ponovno. Pretvorimo datume iz ameriškega zapisa mesec/dan/leto v zapis dan.mesec.leto:

In [15]:
datumi_us = ["01/13/2023", "02/28/2023", "11/10/2023"]

datum_re = re.compile(r"(\d{2})[/-](\d{2})[/-](\d{4})")

datumi_eu = [datum_re.sub(r"\2.\1.\3", d) for d in datumi_us]
print(datumi_eu)

['13.01.2023', '28.02.2023', '10.11.2023']


Vzorec `(\d{2})[/-](\d{2})[/-](\d{4})` zajame tri skupine: **mesec**, **dan** in **leto**. Nadomestni niz `r"\2.\1.\3"` zamenja vrstni red prvih dveh skupin in ločila. S tem pretvorimo zapis, ne preverimo pa koledarske veljavnosti datuma.

Izraz `\w+` uporabimo še za sestavljanje kratic. Kratico sestavimo iz začetnih črk besed, izpustimo besedi `za` in `in` ter črke po potrebi pretvorimo v velike:

In [16]:
def kratica(bz, velike=True, ri=re.compile(r"\w+")):
    vezniki = ["za", "in"]
    k = "".join([b[0] for b in ri.findall(bz) if b.lower() not in vezniki])
    return k.upper() if velike else k


print(kratica("regularni izraz"))
print(kratica("besedna zveza za test"))
print(kratica("Fakulteta za matematiko in fiziko"))

RI
BZT
FMF


Funkcijo `kratica` uporabimo na vsaki vrednosti zaporedja z metodo [`apply`](https://pandas.pydata.org/docs/reference/api/pandas.Series.apply.html). S konstruktorjem [`DataFrame`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html) sestavimo tabelo imen in kratic:

In [17]:
fakultete = pd.DataFrame({
    "ime": fakultete,
    "kratica": fakultete.apply(kratica)
})
print(fakultete)

                                                  ime kratica
0                                 Akademija za glasbo      AG
1   Akademija za gledališče, radio, film in televi...   AGRFT
2        Akademija za likovno umetnost in oblikovanje    ALUO
3                               Biotehniška fakulteta      BF
4                                 Ekonomska fakulteta      EF
5                            Fakulteta za arhitekturo      FA
6                          Fakulteta za družbene vede     FDV
7                         Fakulteta za elektrotehniko      FE
8                              Fakulteta za farmacijo      FF
9              Fakulteta za gradbeništvo in geodezijo     FGG
10        Fakulteta za kemijo in kemijsko tehnologijo    FKKT
11                  Fakulteta za matematiko in fiziko     FMF
12                   Fakulteta za pomorstvo in promet     FPP
13          Fakulteta za računalništvo in informatiko     FRI
14                         Fakulteta za socialno delo     FSD
15      

Za tri članice uporabimo dogovorjene kratice, ki se razlikujejo od rezultata naše funkcije. Vrednosti spremenimo z logičnim indeksiranjem prek atributa [`loc`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.loc.html):

In [18]:
fakultete.loc[fakultete["ime"] == "Fakulteta za farmacijo", "kratica"] = "FFA"
fakultete.loc[fakultete["ime"] == "Fakulteta za šport", "kratica"] = "FSP"
fakultete.loc[fakultete["ime"] == "Pedagoška fakulteta", "kratica"] = "PEF"
print(fakultete)

                                                  ime kratica
0                                 Akademija za glasbo      AG
1   Akademija za gledališče, radio, film in televi...   AGRFT
2        Akademija za likovno umetnost in oblikovanje    ALUO
3                               Biotehniška fakulteta      BF
4                                 Ekonomska fakulteta      EF
5                            Fakulteta za arhitekturo      FA
6                          Fakulteta za družbene vede     FDV
7                         Fakulteta za elektrotehniko      FE
8                              Fakulteta za farmacijo     FFA
9              Fakulteta za gradbeništvo in geodezijo     FGG
10        Fakulteta za kemijo in kemijsko tehnologijo    FKKT
11                  Fakulteta za matematiko in fiziko     FMF
12                   Fakulteta za pomorstvo in promet     FPP
13          Fakulteta za računalništvo in informatiko     FRI
14                         Fakulteta za socialno delo     FSD
15      

Primer prikazuje celoten postopek: prenos spletne strani, izbiro elementov HTML, pretvorbo v podatkovne strukture knjižnice `pandas` in čiščenje podatkov.

Naloge za utrjevanje so v zvezku [Naloge za vaje](urejanje-naloge.ipynb).